# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic dataset metadata (title and description)
print(f"Title: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")

## 2. Data Overview
Let's examine what record sets are available, and the fields within each record set.

All entities such as record sets and fields are referenced by their `@id` in line with the Croissant schema best practices.

In [ ]:
# List all record sets by their `@id`
print("Available Record Sets (by @id):")
record_sets = dataset.metadata.record_sets
record_set_ids = [rs["@id"] for rs in record_sets]
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', rs['@id'])}")

# For each record set, print its fields and their @id
for rs in record_sets:
    print("\nFields in Record Set @id =", rs["@id"])
    for field in rs.get("field", []):
        if isinstance(field, dict):
            f_id = field["@id"]
            name = field.get("name", f_id)
        else:
            f_id = field
            name = f_id
        print(f"  - {f_id}")

## 3. Data Extraction
Let's load and preview a specific record set as a Pandas DataFrame for analysis.

We'll use the `@id` for the record set and field as determined in the previous section. Adjust `PRIMARY_RECORD_SET_ID` as needed based on the available record sets.

In [ ]:
# Pick the main record set to analyze (use @id found above)
# If you are unsure, print(record_set_ids) in the previous cell
PRIMARY_RECORD_SET_ID = record_set_ids[0]
# Optionally, you can select more record sets:
record_sets_to_load = [PRIMARY_RECORD_SET_ID]

dataframes = {}

for record_set_id in record_sets_to_load:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    print(f"\nColumns in DataFrame for record set '{record_set_id}':")
    print(df.columns.tolist())
    print(df.head())
    dataframes[record_set_id] = df

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze numeric and categorical fields.

**Choose a numeric field and an optional group field by their `@id` as shown above.**

This section demonstrates:
- Filtering records based on numeric criteria
- Normalizing numeric fields
- Grouping and aggregating data

In [ ]:
import numpy as np
# --- Choose the correct field IDs ---
# For example, suppose the column for peptide count is '@peptide_count' and sample type is '@sample_id'.
# Use the printed column names from previous cell, or adjust accordingly.
record_set_id = PRIMARY_RECORD_SET_ID
df = dataframes[record_set_id]

# You may need to change these field IDs to match your schema
numeric_field_id = df.columns[df.dtypes.apply(lambda x: np.issubdtype(x, np.number))].tolist()[0]

print(f"Chosen numeric field for analysis: '{numeric_field_id}'")
threshold = df[numeric_field_id].quantile(0.75)  # Use the 75th percentile as threshold
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records with '{numeric_field_id}' > {threshold} (75th percentile):\n")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by a likely candidate categorical column (first object column other than index)
group_candidates = [col for col in df.select_dtypes(include=object).columns if col != df.index.name]
if group_candidates:
    group_field_id = group_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped data by field '{group_field_id}':")
    print(grouped_df.head())
else:
    print("No suitable group (categorical) field for grouping found.")

## 5. Visualization
Visualize field distributions or relationships between variables using Matplotlib or Seaborn.

Below, we'll create a histogram and boxplot for the selected numeric field, and a bar plot by group if grouping is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

# Boxplot
plt.figure(figsize=(8,2))
sns.boxplot(x=df[numeric_field_id])
plt.title(f'Boxplot of {numeric_field_id}')
plt.show()

# Barplot by group (if group candidates exist)
if group_candidates:
    plt.figure(figsize=(10,4))
    order = filtered_df[group_field_id].value_counts().index
    sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df, order=order)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.ylabel(numeric_field_id)
    plt.xlabel(group_field_id)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we've accessed and explored the FAIR² mass spectrometry dataset using the `mlcroissant` library and referenced all schema entities by their `@id` as recommended. We:
- Loaded the metadata and record sets
- Identified and referenced available fields by their `@id`
- Loaded records into a pandas DataFrame for analysis
- Explored, filtered, and normalized a numeric field
- Visualized numeric distributions and group-level summaries (where categorical fields were present)

This workflow provides a repeatable pattern for FAIR Croissant dataset exploration, helping ensure field-level provenance and interpretability for downstream analysis.